## 비교모델 1. LSTM

In [ ]:
import torch
import torch.nn as nn

class LSTMForecast(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=128, num_layers=2, dropout=0.1, output_dim=4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (B, T, input_dim)
        out, _ = self.lstm(x)  # out: (B, T, hidden_dim)
        out = out[:, -1, :]    # 마지막 시점의 hidden state 사용
        out = self.fc(out)     # (B, output_dim)
        return out.unsqueeze(1)  # (B, 1, output_dim) -> [lat, lon, SOG, COG]

# 모델 생성
model = LSTMForecast(input_dim=7, output_dim=4)

# numpy → torch tensor로 변환
input_tensor = torch.tensor(input_seqs, dtype=torch.float32)
output_tensor = torch.tensor(output_seqs, dtype=torch.float32)

# 학습
train_transformer_model(model, (input_tensor, output_tensor), num_epochs=100, device='cuda' if torch.cuda.is_available() else 'cpu')

# 모델 저장
torch.save(model, "lstm_model.pth")

## 비교모델 2. GRU

In [ ]:
import torch
import torch.nn as nn

class GRUForecast(nn.Module):
    def __init__(self, input_dim=7, hidden_dim=128, num_layers=2, dropout=0.1, output_dim=4):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        # x: (B, T, input_dim)
        out, _ = self.gru(x)   # out: (B, T, hidden_dim)
        out = out[:, -1, :]    # 마지막 시점 hidden state
        out = self.fc(out)     # (B, output_dim)
        return out.unsqueeze(1)  # (B, 1, output_dim)

# 모델 생성
model = GRUForecast(input_dim=7, output_dim=4)

# numpy → torch tensor로 변환
input_tensor = torch.tensor(input_seqs, dtype=torch.float32)
output_tensor = torch.tensor(output_seqs, dtype=torch.float32)

# 학습
train_transformer_model(model, (input_tensor, output_tensor), num_epochs=100, device='cuda' if torch.cuda.is_available() else 'cpu')

# 모델 저장
torch.save(model, "gru_model.pth")